In [0]:
%python
print("Prepared by, Crystal Poon")

print("Project: Banking Transaction ETL")
print("Environment: Databricks Free Edition")
print("Language: Python")

In [0]:
%python
print("Spark Version:", spark.version)

Notes:
- Apache Spark = distributed data processing engine 
- Spark able to distribute data processing across compute resources instead of processing large datasets using ordinary Python operations

### Playground

In [0]:
%python
data = [
    (1001, "C001", 500.00, "TRANSFER"),
    (1002, "C002", 250.00, "PAYMENT"),
    (1003, "C003", 1000.00, "TRANSFER")
]

columns = ["transaction_id", "customer_id", "amount", "transaction_type"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
%python
print("TEST 1: Count the transactions")

total_records = df.count()

print("Total transactions:", total_records)

assert total_records == 3, "Transaction count is incorrect"

print("TEST PASSED")

In [0]:
%python
print("TEST 2: Check for missing customer IDs")

from pyspark.sql.functions import col

missing_customer = df.filter(
    col("customer_id").isNull()
).count()

print("Missing customer IDs:", missing_customer)

assert missing_customer == 0, "Missing customer IDs found"

print("TEST PASSED")

In [0]:
%python
print("TEST 3: Check invalid transaction amounts")

invalid_amounts = df.filter(
    col("amount") <= 0
).count()

print("Invalid transaction amounts:", invalid_amounts)

assert invalid_amounts == 0, "Invalid amounts found"

print("TEST PASSED")

### Create the synthetic banking dataset

In [0]:
%python
print("Create the synthetic banking dataset")

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
from pyspark.sql import functions as F

print("Libraries imported successfully")

In [0]:
%python
#StructField(Column name, Data type, Nullable)

transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("ingestion_timestamp", StringType(), True)
])

print("Banking transaction_schema created")

In [0]:
%python
#simulate 20 synthetic records in list 
#(transaction_id, customer_id, amount, transaction_type, transaction_date, source_system, ingestion_timestamp)
transactions = [    
    ("TXN1001", "C001", "500.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1002", "C002", "250.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1003", "C003", "1000.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1004", "C004", "75.00", "WITHDRAWAL", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1005", "C005", "120.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1006", "C006", "0.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1007", None, "80.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1008", "C008", "-50.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1009", "C009", "ABC", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1010", "C010", "45.00", "PAYMENT", "2026-13-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1011", "C011", "200.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1012", "C012", "150.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1013", "C013", "90.00", "WITHDRAWAL", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1014", "C014", "600.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1015", "C015", "80.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1016", "C016", "110.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1017", "C017", "500.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1018", "C018", "220.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1019", "C019", "320.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),

    # Test Duplicate transaction
    ("TXN1002", "C002", "250.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00")
]

print("Total input records:", len(transactions))

In [0]:
%python
print("Convert transaction list into pyspark dataframe")
raw_df = spark.createDataFrame(transactions, schema=transaction_schema)
display(raw_df)

In [0]:
%python
print("Check schema")
raw_df.printSchema()

### Perform Data Profiling

- similar test method use in bronze data validation later

In [0]:
%python
print("Test 1: Count transactions")
total_records = raw_df.count()

print("Total transactions:", total_records)

assert total_records == 20, "Transaction count is incorrect"

print("TEST PASSED")

In [0]:
%python
print("Test 2: Check for duplicate transactions")

from pyspark.sql import functions as F

duplicates_df = (raw_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("transaction_count"))
    .filter("transaction_count > 1")
)

dup_count = duplicates_df.count()

if dup_count > 0:
    display(duplicates_df)  # Only show results when duplicates exist

assert dup_count == 0, f"Found {dup_count} duplicate transaction ID(s)"
       

In [0]:
%python
print("Test 3: Check for invalid Customer id")

invalid_customers_df = raw_df.filter(
    F.col("customer_id").isNull() 
#    | (F.trim(F.col("customer_id")) == "")
#    | F.upper(F.trim(F.col("customer_id"))).isin("NULL", "N/A", "NONE", "UNKNOWN")
    | ~F.col("customer_id").rlike("^C[0-9]{3}$")  # IDs must follow a pattern like CXXX and already able to handle empty strings, "N/A", spaces, and wrong formats
)

missing_customer_count = invalid_customers_df.count()
print(f"Missing customer_id count: {missing_customer_count}")

if missing_customer_count > 0:
    #display(invalid_customers_df)  # or invalid_customers_df.show(truncate=False)
    print("Transaction ID(s) with invalid customer_id:")
    display(invalid_customers_df.select("transaction_id"))

assert missing_customer_count == 0, f"Test Failed: Found {missing_customer_count} invalid customer_id(s)"

print("TEST PASSED")

In PySpark, the tilde symbol (~) is the logical NOT operator used to negate a boolean column condition.

- `^C[0-9]+$` allows any number of digits after C (e.g., C1, C100, C999999)
- `^C[0-9]{3}$` allows only exact length like C001 (3 digits)


In [0]:
%python
from pyspark.sql import functions as F

print("Test 4: Check for invalid or non-numeric amount")

invalid_amount_df = raw_df.filter(
    F.col("amount").isNull()
    | F.expr("try_cast(amount as double)").isNull()  # Safely returns NULL on "ABC" under ANSI mode
)

invalid_amount_count = invalid_amount_df.count()
print(f"Invalid amount count: {invalid_amount_count}")

if invalid_amount_count > 0:
    display(invalid_amount_df.select("transaction_id", "amount"))

assert invalid_amount_count == 0, f"Test Failed: Found {invalid_amount_count} non-numeric amount(s)"

print("TEST PASSED")

### Load Data into Bronze delta table

In [0]:
Select current_catalog(), current_schema()

- Go to Catalog : Manually create new Catalog "banking_demo"
- Click on banking_demo, manually create 3 new schemas : bronze, silver, gold

In [0]:
USE CATALOG banking_demo;
USE SCHEMA bronze;

Select current_catalog(), current_schema()

In [0]:
%python
print("Set current location")
spark.catalog.setCurrentCatalog("banking_demo")
spark.catalog.setCurrentDatabase("bronze")  # "Database" is Spark's alias for Schema

print("Check current location")
print(f"Catalog: {spark.catalog.currentCatalog()}, Schema: {spark.catalog.currentDatabase()}")

In [0]:
%python
#double check table to populate into bronze table
display(raw_df)

In [0]:
%python
print("Create new Bronze table, transaction")
catalog_name = spark.catalog.currentCatalog()
schema_name = spark.catalog.currentDatabase()
bronze_table_name = "transaction"

raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{schema_name}.{bronze_table_name}")  

print("Table created")

In [0]:
select * from transaction;

In [0]:
%python
# Read the table
bronze_df = spark.read.table(f"{catalog_name}.{schema_name}.{bronze_table_name}")
display(bronze_df)
  

In [0]:
%python
# Validate bronze table
print("Test B1: Count of records in bronze table")

bronze_transaction_count = bronze_df.count()
print(f"Bronze transaction count: {bronze_transaction_count}")

assert bronze_transaction_count == 20, f"Test Failed: Found {bronze_transaction_count} records in bronze table"

print("TEST PASSED")

In [0]:
%python
# Validate bronze table
print("Test B2: Check for duplicate transactions")

from pyspark.sql import functions as F

duplicates_bronze_df = (bronze_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("transaction_count"))
    .filter("transaction_count > 1")
)

dup_count = duplicates_bronze_df.count()

if dup_count > 0:
    display(duplicates_bronze_df)  # Only show results when duplicates exist

assert dup_count == 0, f"Found {dup_count} duplicate transaction ID(s)"
       

In [0]:
%python
# Validate bronze table
print("Test B3: Check for invalid Customer id")

invalid_customers_df = bronze_df.filter(
    F.col("customer_id").isNull() 
#    | (F.trim(F.col("customer_id")) == "")
#    | F.upper(F.trim(F.col("customer_id"))).isin("NULL", "N/A", "NONE", "UNKNOWN")
    | ~F.col("customer_id").rlike("^C[0-9]{3}$")  # IDs must follow a pattern like CXXX and already able to handle empty strings, "N/A", spaces, and wrong formats
)

missing_customer_count = invalid_customers_df.count()
print(f"Missing customer_id count: {missing_customer_count}")

if missing_customer_count > 0:
    #display(invalid_customers_df)  # or invalid_customers_df.show(truncate=False)
    print("Transaction ID(s) with invalid customer_id:")
    display(invalid_customers_df.select("transaction_id"))

assert missing_customer_count == 0, f"Test Failed: Found {missing_customer_count} invalid customer_id(s)"

print("TEST PASSED")

In [0]:
%python
# Validation bronze table
from pyspark.sql import functions as F

print("Test 4: Check for invalid or non-numeric amount")

invalid_amount_df = bronze_df.filter(
    F.col("amount").isNull()
    | F.expr("try_cast(amount as double)").isNull()  # Safely returns NULL on "ABC" under ANSI mode
)

invalid_amount_count = invalid_amount_df.count()
print(f"Invalid amount count: {invalid_amount_count}")

if invalid_amount_count > 0:
    display(invalid_amount_df.select("transaction_id", "amount"))

assert invalid_amount_count == 0, f"Test Failed: Found {invalid_amount_count} non-numeric amount(s)"

print("TEST PASSED")